# Appendix — LlamaIndex Workflows: fixed control flow as events

Seven framework appendices run the *same* Larkspur triage — ticket `TKT-2205`, gold `partial_refund | pol-restocking | $170.99` — so you can compare frameworks on one fixed problem. This is the **workflow** archetype, the deliberate contrast to the agent-loop appendices: [LlamaIndex Workflows](https://developers.llamaindex.ai/python/llamaagents/workflows/) is "an event-driven, step-based way to control the execution flow of an application." The steps and their order are *code you wrote*; the model reasons inside one step. That is your chapter 05 workflow — routing and prompt-chaining, fixed control flow — not the chapter 02 loop where the model decides what happens next.

> **Before running this notebook:** `pip install -e ".[llamaindex]"` (once). It pulls in `llama-index-workflows` and `llama-index-llms-litellm`, which co-install with the main venv — no separate kernel. The model reaches the same OpenRouter endpoint you have used since ch01, through LiteLLM, so the disk cache still answers reruns. Everything else stays the same.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

## The model boundary: a LiteLLM handle

A workflow step that needs the model asks LlamaIndex for an `LLM` object. `llama_index.llms.litellm.LiteLLM` wraps the very LiteLLM you have called since chapter 01, so the model string is the course default `openrouter/deepseek/deepseek-v3.2` and calls still pass through the disk cache — a rerun is ~free. We set `temperature=0` on the handle to keep the course's determinism. This is the chapter 01 model boundary again, wearing LlamaIndex's `LLM` interface.

In [ ]:
from llama_index.llms.litellm import LiteLLM
from llama_index.core.llms import ChatMessage

llm = LiteLLM(model=MODEL, temperature=TEMPERATURE)   # same LiteLLM, same cache, same key
print("llm  :", type(llm).__module__ + "." + type(llm).__name__)
print("model:", llm.model)

## The verdict is pinned to gold

Every appendix must land on the same economics, so pin the invariant now. The authoritative gold comes from `shoplab.rules.decide` for `TKT-2205` — an opened, in-window return from a non-vip member, so branch 9 of the cascade charges a 10% restocking fee: `$189.99 x 0.90`. The workflow below must reach exactly this.

In [ ]:
from shoplab import world, rules

orders = {o["order_id"]: o for o in world.load_orders()}
customers = {c["customer_id"]: c for c in world.load_customers()}
t2205 = next(t for t in world.load_tickets()["train"] if t["ticket_id"] == "TKT-2205")
GOLD = rules.decide(t2205, orders[t2205["order_id"]], customers[t2205["customer_id"]])
print("authoritative gold (shoplab.rules.decide):", GOLD)   # the invariant every appendix hits

## Tools are functions the steps call

Here is the fork in the road. In the agent-loop appendices the model is handed a tool registry and *chooses* which to call; that is chapter 02. A workflow inverts it: the four Larkspur operations are plain functions, and *your step code* decides when each runs. So `get_order`, `search_policy`, `calc`, and `issue_refund` are thin wrappers that import from `shoplab.world` / `shoplab.tools` — never reimplementations — and `issue_refund` writes to chapter 02's real `Ledger` so we can watch the side effect. No JSON tool schema, no tool-selection prompt: the control flow is code.

In [ ]:
from shoplab.tools import calc, Ledger

ledger = Ledger()                              # ch02's real side-effect log

def get_order(order_id):
    """Look up a Larkspur order (items, totals, status)."""
    return orders.get(order_id, {"error": f"no such order {order_id}"})

def search_policy(query, k=2):
    """Keyword-search the 12 Larkspur policy documents."""
    return world.search_policy(query, k=k)

def issue_refund(order_id, amount_usd, reason):
    """Send money back to the customer. Irreversible -- writes the Ledger."""
    rid = f"REF-{1001 + len(ledger.entries)}"
    ledger.record("issue_refund", refund_id=rid, order_id=order_id,
                  amount_usd=amount_usd, reason=reason)
    return {"ok": True, "refund_id": rid}

print("wrappers ready: get_order, search_policy, calc, issue_refund (+ Ledger)")

## The workflow: three steps, wired by events

A `Workflow` is a class of `@step` methods; each step receives one event and returns the next, and LlamaIndex routes them by type. Our fixed flow is exactly chapter 05's prompt-chaining — `lookup` gathers facts, `decide` calls the model, `finalize` acts — but expressed as events instead of nested calls. `lookup` (on the `StartEvent`) pulls the order and policy deterministically; `decide` hands those facts to the model and computes the fee with `calc`; the run then pauses at the gate (next section) before `finalize` moves the money and returns the `StopEvent`.

In [ ]:
from llama_index.core.workflow import (
    Workflow, step, Event, StartEvent, StopEvent, Context,
    InputRequiredEvent, HumanResponseEvent,
)
import json

class DecideEvent(Event):
    """lookup -> decide: facts gathered, ready to adjudicate."""

class TriageFlow(Workflow):
    @step
    async def lookup(self, ctx: Context, ev: StartEvent) -> DecideEvent:
        t = next(x for x in world.load_tickets()["train"] if x["ticket_id"] == ev.ticket_id)
        order = get_order(t["order_id"])
        line = next(i for i in order["items"] if i["sku"] == t["sku"])
        await ctx.store.set("ticket", t)
        await ctx.store.set("item_value", round(t["qty"] * line["unit_price_usd"], 2))
        await ctx.store.set("tier", customers[t["customer_id"]]["tier"])
        await ctx.store.set("policies", search_policy(t["reason_text"], k=2))
        return DecideEvent()

    @step
    async def decide(self, ctx: Context, ev: DecideEvent) -> InputRequiredEvent:
        t = await ctx.store.get("ticket")
        item_value = await ctx.store.get("item_value")
        tier = await ctx.store.get("tier")
        pols = await ctx.store.get("policies")
        pol_text = "\n".join(f"{p['id']}: {p['text']}" for p in pols)
        prompt = (
            "You are the Larkspur ops desk. Reply ONLY as JSON "
            '{"decision": "...", "policy_id": "pol-...", "refund_expr": "<arithmetic or null>"}.\n'
            f"condition={t['item_condition']} action={t['requested_action']} "
            f"days_since_delivery={t['days_since_delivery']} tier={tier} item_value={item_value}\n"
            f"Policies:\n{pol_text}\n"
            "An opened change-of-mind refund inside 30 days is partial_refund under pol-restocking "
            "with a 10% restocking fee (waived only for vip). Put the fee arithmetic in refund_expr."
        )
        raw = llm.chat([ChatMessage(role="user", content=prompt)]).message.content
        d = json.loads(raw[raw.find("{"): raw.rfind("}") + 1])
        expr = d.pop("refund_expr", None)
        d["refund_usd"] = round(calc(expr), 2) if expr and expr != "null" else None
        await ctx.store.set("proposal", d)
        return InputRequiredEvent(
            prefix=f"Approve issue_refund({d['refund_usd']}, {t['order_id']})? [y/n] ")

    @step
    async def finalize(self, ctx: Context, ev: HumanResponseEvent) -> StopEvent:
        d = await ctx.store.get("proposal")
        t = await ctx.store.get("ticket")
        if ev.response.strip().lower().startswith("y") and d["refund_usd"] is not None:
            issue_refund(t["order_id"], d["refund_usd"], "opened return: 10% restocking fee")
        return StopEvent(result=d)

print("TriageFlow steps: lookup -> decide -> (gate) -> finalize")

## The approval gate is a native pause

Chapter 08 wrapped risky tools in `require_approval` so money moved only after a human said yes. LlamaIndex ships that as a first-class event pair. When `decide` returns an `InputRequiredEvent`, the run *stops* and surfaces it on the event stream; nothing calls `issue_refund` yet. A human (here, our code) answers with a `HumanResponseEvent`, which is the only thing that triggers `finalize`. So the gate is structural: the side-effect step literally cannot run until the response event arrives. Run the triage and watch the `Ledger` stay at `0` at the gate, then tick to `1` once approved.

In [ ]:
async def run_triage(ticket_id, approve=True):
    handler = TriageFlow(timeout=120).run(ticket_id=ticket_id)
    async for ev in handler.stream_events():
        if isinstance(ev, InputRequiredEvent):
            print("GATE:", ev.prefix.strip(), "| ledger at gate:", len(ledger.entries))
            handler.ctx.send_event(HumanResponseEvent(response="yes" if approve else "no"))
    return await handler

verdict = await run_triage("TKT-2205")
print("verdict       :", verdict)
print("ledger entries:", len(ledger.entries), "(the approved write happened)")
print("matches gold  :", verdict == GOLD)

> **What you should see:** at the gate the `Ledger` reads `0` — `decide` proposed `issue_refund(170.99, ORD-7312)` but the `InputRequiredEvent` pause meant no money moved. Sending the `HumanResponseEvent` releases `finalize`, the `Ledger` ticks to `1`, and the workflow returns `partial_refund | pol-restocking | 170.99`. `matches gold` is `True`: the fixed-flow workflow landed on the same economics as `shoplab.rules.decide` (`$189.99 x 0.90`), the invariant every appendix in this set shares.

## Machinery map: LlamaIndex Workflows to the part you built

Line them up and the framework stops being magic. Every Workflows concept here maps to a piece of fixed-flow machinery you built by hand in chapter 05 (with the model boundary and gate from chapters 01 and 08).

| LlamaIndex Workflows concept | Your hand-built equivalent | Built in |
|---|---|---|
| `Workflow` subclass, fixed step graph | routing / prompt-chaining pipeline: control flow you own | ch05 |
| `@step` methods routed by event type | the chain's `extract -> gate -> decide` stages | ch05 |
| `Event` subclasses passed between steps | the intermediate dict handed from one stage to the next | ch05 |
| `StartEvent` / `StopEvent` | the pipeline's input ticket and returned decision | ch05 |
| `Context.store` (`ctx.store.set/get`) | local variables threaded through the chain | ch05 |
| `LiteLLM` handle inside `decide` | `shoplab.llm.complete` wrapping the model boundary | ch01 |
| `InputRequiredEvent` / `HumanResponseEvent` | `require_approval` pausing a risky tool for a yes/no | ch08 |

## The honest trade

Workflows buy you *explicit* control flow. Unlike the agent-loop appendices, nothing here decides the plan at runtime: `lookup` always precedes `decide`, and the model is confined to one step where it adjudicates the facts you fed it — so the workflow cannot wander off, call a tool you did not intend, or loop. The event routing makes the shape legible: to see what runs when you read the `@step` return types, not a trace. And the approval gate is genuinely structural, not a convention — `finalize` is unreachable without a `HumanResponseEvent`.

The cost is that fixed flow is *fixed*. This workflow only ever triages one opened-refund shape; a ticket needing a different investigation would need new steps and events, where the chapter 02 agent would just call different tools in a different order. You also write more scaffolding — event classes, a `Context` store, an async stream loop — to express what chapter 05's `chain` said in a nested function call. And the seam moved as always: the model call goes through LlamaIndex's `LLM` object, so anything you instrumented on `shoplab.llm.complete` now needs to hook LiteLLM instead. Workflows is the right tool when the process is known and you want it pinned down; it is the wrong one when you want the model to find the process.